In [8]:
import json
import pandas as pd
import networkx as nx
from pathlib import Path
from collections import Counter

RESULTS_DIR  = Path("../data/results")
ATTCK_DIR    = Path("../data/attck")
STIX_FILE    = ATTCK_DIR   / "enterprise-attack.json"
TRIPLES_FILE = RESULTS_DIR / "community_triples.json"
GRAPH_FILE   = RESULTS_DIR / "knowledge_graph.graphml"
NODE_FILE    = RESULTS_DIR / "knowledge_graph_nodes.csv"
EDGE_FILE    = RESULTS_DIR / "knowledge_graph_edges.csv"
METRICS_FILE = RESULTS_DIR / "knowledge_graph_metrics.json"

with open(TRIPLES_FILE, "r", encoding="utf-8") as f:
    community_triples = json.load(f)

print(f"Loaded triples from {len(community_triples)} communities")
print("Sample:", list(community_triples.items())[0])

Loaded triples from 19 communities
Sample: ('0', [{'subject': 'ftp_brute_force_client', 'relation': 'BRUTE_FORCES_CREDENTIAL', 'target': 'ftp_authentication_service'}, {'subject': 'http_scanner', 'relation': 'PERFORMS_RECONNAISSANCE', 'target': 'http_web_server'}, {'subject': 'http_scanner', 'relation': 'ESTABLISHES_C2', 'target': 'command_and_control_channel'}, {'subject': 'http_scanner', 'relation': 'EXFILTRATES_DATA', 'target': 'internal_data_store'}])


In [9]:
# Load ATT&CK technique definitions from the STIX bundle
#
# These technique nodes serve as a structured dictionary anchored to MITRE ATT&CK.
#
# DESIGN RATIONALE
# ----------------
# Adding technique nodes does NOT constitute data leakage because no community
# is pre-labeled with a technique ID. The nodes provide a vocabulary that the
# RAG stage can traverse toward, but the graph still has to derive the path
# from extracted community triples to a technique node through relation matching.
#
# WHY ALL TECHNIQUES ARE LOADED (no tactic filter)
# -------------------------------------------------
# In real SOC deployment there is no prior knowledge of which tactics or
# techniques will appear in the incoming alert stream. Filtering techniques
# to only those tactics present in our dataset would mean using dataset-level
# knowledge to shape the graph — a form of information leakage that would
# artificially inflate performance and would not generalise.
#
# Loading all 700+ techniques keeps the graph honest: the pipeline has to
# find the correct technique from the full ATT&CK catalogue, exactly as a
# real system would. The graph will be larger, but the evaluation will be
# a fair measure of the method's actual capability.

with open(STIX_FILE, "r", encoding="utf-8") as f:
    bundle = json.load(f)

attck_techniques = {}
for obj in bundle.get("objects", []):
    if obj.get("type") != "attack-pattern" or obj.get("revoked", False):
        continue

    tid = None
    for ref in obj.get("external_references", []):
        if ref.get("source_name") == "mitre-attack":
            tid = ref.get("external_id")
            break
    if not tid:
        continue

    tactics = [
        p["phase_name"]
        for p in obj.get("kill_chain_phases", [])
        if p.get("kill_chain_name") == "mitre-attack"
    ]

    attck_techniques[tid] = {
        "name":        obj.get("name", "Unknown"),
        "tactic":      tactics[0].replace("-", " ").title() if tactics else "Unknown",
        "description": obj.get("description", "")[:300],  # truncated to keep node properties readable
    }

all_tactics = {v["tactic"] for v in attck_techniques.values()}
print(f"Loaded {len(attck_techniques)} ATT&CK techniques across {len(all_tactics)} tactics (full catalogue)")

Loaded 703 ATT&CK techniques across 14 tactics (full catalogue)


In [10]:
# Graph construction
#
# The graph has two layers:
#
# 1. BEHAVIORAL LAYER  (community triples)
#    Nodes and edges derived from LLM extraction in Stage 2.
#    Edge weights encode how often a triple appears across communities,
#    emphasising recurring patterns over noise.
#
# 2. KNOWLEDGE LAYER   (ATT&CK technique nodes)
#    Nodes representing MITRE technique definitions. These are added as
#    standalone dictionary nodes. Edges connecting them to behavioral nodes
#    are built in the next cell through relation matching.
#
# This separation makes it easy to distinguish observed behaviour from
# curated threat intelligence when inspecting the graph.

G = nx.DiGraph()

# -- Behavioral layer --
edge_weights     = {}
edge_communities = {}
total_triples    = 0
invalid_triples  = 0

for cid, triples in community_triples.items():
    for t in triples:
        s = str(t.get("subject",  "")).strip()
        r = str(t.get("relation", "")).strip()
        o = str(t.get("target",   "")).strip()

        if not s or not r or not o:
            invalid_triples += 1
            continue

        total_triples += 1
        key = (s, r, o)
        edge_weights[key]       = edge_weights.get(key, 0) + 1
        edge_communities.setdefault(key, []).append(cid)

for (src, rel, tgt), weight in edge_weights.items():
    G.add_node(src, layer="behavioral")
    G.add_node(tgt, layer="behavioral")
    G.add_edge(
        src, tgt,
        relation=rel,
        weight=weight,
        layer="behavioral",
        communities=",".join(edge_communities[(src, rel, tgt)])
    )

# -- Knowledge layer --
for tid, info in attck_techniques.items():
    G.add_node(
        tid,
        layer="attck",
        name=info["name"],
        tactic=info["tactic"],
        description=info["description"]
    )

print(f"Graph constructed: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print(f"Behavioral nodes:  {sum(1 for n, d in G.nodes(data=True) if d.get('layer') == 'behavioral')}")
print(f"ATT&CK nodes:      {sum(1 for n, d in G.nodes(data=True) if d.get('layer') == 'attck')}")
print(f"Processed {total_triples} triples | Skipped {invalid_triples} invalid")

Graph constructed: 752 nodes, 53 edges
Behavioral nodes:  49
ATT&CK nodes:      703
Processed 76 triples | Skipped 0 invalid


In [ ]:
# Bridge edges: connect behavioral nodes to ATT&CK technique nodes

RELATION_TO_TECHNIQUES = {
    "PERFORMS_RECONNAISSANCE":  ["T1046",    "T1595",    "T1590"],
    "BRUTE_FORCES_CREDENTIAL":  ["T1110",    "T1110.001","T1110.003"],
    "EXPLOITS_VULNERABILITY":   ["T1190",    "T1203"],
    "ESTABLISHES_C2":           ["T1071",    "T1071.001","T1071.004"],
    "CAUSES_DENIAL_OF_SERVICE": ["T1498",    "T1499",    "T1499.001"],
    "MOVES_LATERALLY":          ["T1021",    "T1570"],
    "EXFILTRATES_DATA":         ["T1041",    "T1048"],
    "EXECUTES_PAYLOAD":         ["T1059",    "T1059.007","T1105"],
}

bridge_edges_added = 0

for (src, rel, tgt), _ in edge_weights.items():
    candidate_techniques = RELATION_TO_TECHNIQUES.get(rel, [])
    for tid in candidate_techniques:
        if tid in G:  # only link to techniques that exist in our filtered set
            G.add_edge(
                src, tid,
                relation="ASSOCIATED_WITH",
                weight=1,
                layer="bridge",
                communities=""
            )
            bridge_edges_added += 1

print(f"Bridge edges added: {bridge_edges_added}")
print(f"Total graph edges:  {G.number_of_edges()}")

Bridge edges added: 164
Total graph edges:  175


In [12]:
# Graph analysis
# Inspect the top behavioral nodes and recurring edge patterns.
# ATT&CK technique nodes are excluded from the degree ranking here
# because their degree is dominated by bridge edges, not observed behaviour.

behavioral_nodes = [(n, d) for n, d in G.degree() if G.nodes[n].get("layer") == "behavioral"]

print("TOP 10 BEHAVIORAL NODES BY DEGREE")
for node, deg in sorted(behavioral_nodes, key=lambda x: x[1], reverse=True)[:10]:
    print(f"  {node} (degree={deg})")

print("\nTOP 10 BEHAVIORAL EDGES BY WEIGHT")
behavioral_edges = [
    (u, v, d["relation"], d["weight"])
    for u, v, d in G.edges(data=True)
    if d.get("layer") == "behavioral"
]
for u, v, rel, w in sorted(behavioral_edges, key=lambda x: x[3], reverse=True)[:10]:
    print(f"  (weight={w}) {u} --[{rel}]--> {v}")

print("\nATT&CK NODES REACHABLE FROM BEHAVIORAL LAYER")
reachable_attck = [
    n for n in G.nodes()
    if G.nodes[n].get("layer") == "attck" and G.in_degree(n) > 0
]
print(f"  {len(reachable_attck)} technique nodes have at least one incoming bridge edge")
for tid in sorted(reachable_attck)[:10]:
    info = G.nodes[tid]
    print(f"  {tid} | {info.get('name', '')} | {info.get('tactic', '')}")

TOP 10 BEHAVIORAL NODES BY DEGREE
  attacker (degree=27)
  https_client (degree=15)
  http_scanner (degree=11)
  dns_scanner (degree=11)
  dns_query_client (degree=9)
  unknown_port_1723 (degree=9)
  ftp_brute_force_client (degree=8)
  ssh_scanner (degree=8)
  http_alt_client (degree=8)
  dns_client (degree=8)

TOP 10 BEHAVIORAL EDGES BY WEIGHT
  (weight=5) ftp_brute_force_client --[BRUTE_FORCES_CREDENTIAL]--> ftp_authentication_service
  (weight=4) ssh_scanner --[PERFORMS_RECONNAISSANCE]--> ssh_port_22_service
  (weight=4) attacker --[ESTABLISHES_C2]--> unknown
  (weight=2) attacker --[ESTABLISHES_C2]--> command_and_control_channel
  (weight=2) dns_brute_force_client --[PERFORMS_RECONNAISSANCE]--> dns_service
  (weight=2) dns_query_client --[PERFORMS_RECONNAISSANCE]--> dns_server
  (weight=2) credential_guessing_process --[EXECUTES_PAYLOAD]--> password_spray_module
  (weight=1) ftp_brute_force_client --[PERFORMS_RECONNAISSANCE]--> ftp_authentication_endpoint
  (weight=1) http_scanner 

In [13]:
# Export graph structure
# GraphML preserves all node and edge attributes including the layer field,
# which distinguishes behavioral nodes from ATT&CK technique nodes.
nx.write_graphml(G, GRAPH_FILE)

# Node export: include layer and ATT&CK metadata where available
nodes_out = []
for n, d in G.nodes(data=True):
    nodes_out.append({
        "node":        n,
        "degree":      G.degree(n),
        "layer":       d.get("layer",       ""),
        "name":        d.get("name",        ""),
        "tactic":      d.get("tactic",      ""),
        "description": d.get("description", ""),
    })
pd.DataFrame(nodes_out).sort_values("degree", ascending=False).to_csv(NODE_FILE, index=False)

# Edge export
edges_out = [
    {
        "source":      u,
        "target":      v,
        "relation":    data["relation"],
        "weight":      data["weight"],
        "layer":       data.get("layer",       ""),
        "communities": data.get("communities", ""),
    }
    for u, v, data in G.edges(data=True)
]
pd.DataFrame(edges_out).sort_values("weight", ascending=False).to_csv(EDGE_FILE, index=False)
print(f"Graph exported to {GRAPH_FILE}")

Graph exported to ../data/results/knowledge_graph.graphml


In [14]:
# Compute and save graph metrics
relation_dist = Counter([data["relation"] for _, _, data in G.edges(data=True)])

metrics = {
    "construction_method":       "LLM schema-based extraction (security ontology) -> NetworkX DiGraph",
    "extraction_model":          "qwen2.5-8b-instruct-q4_k_m",
    "relation_ontology":         "8-relation security schema (PERFORMS_RECONNAISSANCE, BRUTE_FORCES_CREDENTIAL, ...)",
    "attck_catalogue":            "full (all non-revoked techniques, no tactic filter)",
    "total_triples_processed":   total_triples,
    "invalid_triples_skipped":   invalid_triples,
    "behavioral_nodes":          sum(1 for n, d in G.nodes(data=True) if d.get("layer") == "behavioral"),
    "attck_technique_nodes":     sum(1 for n, d in G.nodes(data=True) if d.get("layer") == "attck"),
    "total_nodes":               G.number_of_nodes(),
    "behavioral_edges":          sum(1 for _, _, d in G.edges(data=True) if d.get("layer") == "behavioral"),
    "bridge_edges":              sum(1 for _, _, d in G.edges(data=True) if d.get("layer") == "bridge"),
    "total_edges":               G.number_of_edges(),
    "average_degree":            round(sum(dict(G.degree()).values()) / max(G.number_of_nodes(), 1), 3),
    "relation_distribution":     dict(relation_dist),
    "attck_nodes_reachable":     len(reachable_attck),
}

with open(METRICS_FILE, "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

print("KNOWLEDGE GRAPH METRICS")
for k, v in metrics.items():
    print(f"  {k}: {v}")

KNOWLEDGE GRAPH METRICS
  construction_method: LLM schema-based extraction (security ontology) -> NetworkX DiGraph
  extraction_model: qwen2.5-8b-instruct-q4_k_m
  relation_ontology: 8-relation security schema (PERFORMS_RECONNAISSANCE, BRUTE_FORCES_CREDENTIAL, ...)
  attck_catalogue: full (all non-revoked techniques, no tactic filter)
  total_triples_processed: 76
  invalid_triples_skipped: 0
  behavioral_nodes: 49
  attck_technique_nodes: 703
  total_nodes: 752
  behavioral_edges: 53
  bridge_edges: 122
  total_edges: 175
  average_degree: 0.465
  relation_distribution: {'BRUTE_FORCES_CREDENTIAL': 7, 'PERFORMS_RECONNAISSANCE': 15, 'ASSOCIATED_WITH': 122, 'ESTABLISHES_C2': 15, 'EXFILTRATES_DATA': 5, 'EXPLOITS_VULNERABILITY': 8, 'MOVES_LATERALLY': 1, 'EXECUTES_PAYLOAD': 2}
  attck_nodes_reachable: 18
